# Local Whisper w Colab — warsztat BrainHack 2026

Część warsztatowa modułu **4 (Ochrona danych i modele lokalne)**.

Uruchamiaj komórki kolejno (`Shift+Enter`). Własne nagranie (~1 min) wgraj w sekcji „1. Wgraj własne nagranie”.
Model działa w sesji Colab — audio trafia do maszyny Google, ale **nie** do zewnętrznych API dostawców ASR.


In [ ]:
# 0. Instalacja
!pip install -q "av<19" faster-whisper jiwer
import os


In [ ]:
# 1. Wgraj własne nagranie (~1 min, format mp3/wav/m4a)
from google.colab import files
uploaded = files.upload()
AUDIO = list(uploaded.keys())[0]
print('Wgrano:', AUDIO)


In [ ]:
# Odsłuch nagrania
from IPython.display import Audio, display
display(Audio(AUDIO))


## 2. Transkrypcja (speech-to-text) — faster-whisper

Model działa lokalnie w sesji Colab. Wynik to lista segmentów z czasami i tekstem.

Używamy mniejszego modelu **`small`** — wystarczającego na potrzeby prezentacji/dema. Z tego powodu wyniki (jakość transkrypcji) mogą być mniej dokładne niż dla większych modeli.


In [ ]:
from faster_whisper import WhisperModel

model = WhisperModel('small', device='cpu', compute_type='int8')  # GPU: device='cuda', compute_type='float16'
segments, info = model.transcribe(AUDIO, language='pl')
segments = list(segments)

transcript = '\n'.join(s.text.strip() for s in segments)

for s in segments:
    print(f'[{s.start:6.1f}s -> {s.end:6.1f}s] {s.text.strip()}')

print('\n--- PEŁNA TRANSKRYPCJA ---')
print(transcript)


## 3. Zapisanie do pliku (txt + SRT)

Zapisujemy transkrypcję jako zwykły tekst i jako napisy SRT, potem pobieramy oba pliki.


In [ ]:
from google.colab import files

base = os.path.splitext(AUDIO)[0]

# zwykły tekst
with open(f'{base}_transkrypcja.txt', 'w', encoding='utf-8') as f:
    f.write(transcript)

# SRT (napisy)
def srt_time(t):
    h, rem = divmod(t, 3600)
    m, s = divmod(rem, 60)
    return f'{int(h):02}:{int(m):02}:{int(s):02},{int((s % 1) * 1000):03}'

with open(f'{base}.srt', 'w', encoding='utf-8') as f:
    for i, s in enumerate(segments, 1):
        f.write(f'{i}\n{srt_time(s.start)} --> {srt_time(s.end)}\n{s.text.strip()}\n\n')

print('Zapisano:', f'{base}_transkrypcja.txt', 'i', f'{base}.srt')
files.download(f'{base}_transkrypcja.txt')
files.download(f'{base}.srt')


## 4. Opcjonalnie: metryki WER / CER

WER/CER liczymy względem **Twojej** ręcznej transkrypcji **Twojego** nagrania: wklej ją do zmiennej `reference`. Jeśli `reference` jest puste, komórka pomija metryki.


In [ ]:
from jiwer import wer, cer

reference = ''  # wklej własną, ręczną transkrypcję swojego nagrania


if reference.strip():
    print(f'WER: {wer(reference, transcript):.1%}')
    print(f'CER: {cer(reference, transcript):.1%}')
else:
    print('Brak referencji — pomijam WER/CER. Uzupełnij zmienną `reference`, aby policzyć metryki.')


In [ ]:
# Wizualizacja CER w HTML — różnice znak po znaku
import difflib
from html import escape
from IPython.display import HTML, display

if reference.strip():
    RED = 'background:#680000;text-decoration:line-through'
    GREEN = 'background:#006b00'
    sm = difflib.SequenceMatcher(None, reference, transcript, autojunk=False)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == 'equal':
            parts.append(escape(reference[i1:i2]))
            continue
        if i2 > i1:
            parts.append(f'<span style="{RED}">{escape(reference[i1:i2])}</span>')
        if j2 > j1:
            parts.append(f'<span style="{GREEN}">{escape(transcript[j1:j2])}</span>')
    body = ''.join(parts)
    cer_score = cer(reference, transcript)
    display(HTML(f'''
    <div style="font-family:sans-serif;max-width:800px">
      <h3>CER: {cer_score:.1%}</h3>
      <p><span style="{RED}">czerwony</span> = brakuje / błąd (referencja) &nbsp;
         <span style="{GREEN}">zielony</span> = dodatkowe / błędne (transkrypcja)</p>
      <div style="line-height:1.8;white-space:pre-wrap;border:1px solid #ccc;padding:10px;border-radius:6px">{body}</div>
    </div>'''))
else:
    print('Brak referencji — pomijam wizualizację CER.')


## 5. Podsumowanie

Wyprodukowane pliki: `*_transkrypcja.txt` (tekst) i `*.srt` (napisy). Możesz je pobrać z panelu plików Colab.

Prywatność: model działa w sesji Colab — audio nie trafia do zewnętrznych API ASR, ale jest przesyłane do maszyny Google (Colab). Dla pełnej lokalności uruchom Whisper na własnym komputerze.
